# Meet the Loop: talk to Gemini Pupper, no robot needed

CS 123, Lab 6 (*Bringing Your Pupper to Life*), Part 0.

This notebook runs `lab/chat.py` from the Gemini Pupper repo: **your** prompt and **your** tools, the exact list the web app gets, on a **fake robot**. You type, Gemini answers, and every tool call, every reply, and everything the fake robot did is printed. Nothing moves, and you don't need a Pupper.

The chat is based on a short Gemini Live example by Peng Xu (Google DeepMind), who created Gemini Pupper. Thank you, Peng!

## 1. Your Gemini API key

Create a key in [Google AI Studio](https://aistudio.google.com/apikey). Then, in Colab:

1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add new secret**: name `GEMINI_API_KEY`, value: your key.
3. Turn on **Notebook access** for it.

Never paste the key into a cell: cells get saved, shared and submitted.

## 2. Get the code

By default this clones the class repo. To try **your own** code, put your fork's URL in `REPO_URL` (a public fork, or see the handout for a private one).

In [ ]:
REPO_URL = "https://github.com/cs123-stanford/gemini_pupper.git"  # @param {type:"string"}

import os, subprocess
os.chdir("/content")
if not os.path.exists("/content/gemini_pupper"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/gemini_pupper"], check=True)
subprocess.run(["pip", "install", "-q", "google-genai", "pyyaml"], check=True)
os.chdir("/content/gemini_pupper")
print("ready:", os.getcwd())

In [ ]:
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("key loaded from Colab Secrets")
except Exception as e:
    print("No key yet: add GEMINI_API_KEY in Secrets (key icon, left), with notebook access on.", type(e).__name__)

## 3. Talk to it

Run the cell below, and type into the box that appears:

> lie down, then stand up, then walk forward

You will see something like this:

```text
You > lie down, then stand up, then walk forward
  -> Gemini calls lie_down({})
  <- reply Gemini hears now: {"success": true, "message": "deactivate queued; the queue should finish in 1.0 s"}
     ...later, when the queue gets to it:
     [fake robot] legs: relaxed gemini_walk
     [fake robot] outcome: deactivate: ok
  -> Gemini calls stand_up({})
  ...
  -> Gemini calls move({"direction": "forward"})
  <- reply Gemini hears now: {"success": false, "message": "TODO 3 is not written yet (move in lab/part2_walk.py)"}
Pupper > ...
```

That is the whole loop the lab is about. Gemini never runs code: it reads the description of each tool (the `@tool(...)` lines in `lab/`), asks for a call, and the code runs it and sends back a reply. **The reply is all Gemini ever learns.** Notice, too, that the *outcome* (`...later`) is printed for you, not sent to Gemini.

Type `quit` to end the chat (or press the stop button).

In [ ]:
!python3 -u lab/chat.py

## 4. Two experiments

Edit `lab/basics.py` right here in Colab: open the **Files** panel (folder icon, left), go to `gemini_pupper/lab/basics.py`, double-click it, edit, and save (Ctrl+S). Then run the chat cell again: `chat.py` reads `lab/` when it starts.

The cell below shows the two tools you will change.

In [ ]:
!sed -n '/^@tool("stand_up"/,/^async def lie_down_tool/p' lab/basics.py

**a. The words drive the call.** Ask Pupper to *"take a nap"*. Then delete the sentence *"Use when the user says sleep, relax or lie down."* from `lie_down`'s description (it spans two lines) and ask again. Last, change the description to `"Makes the robot jump."` and ask it to jump. What does Gemini call each time?

**b. The reply is all Gemini learns.** Make `stand_up_tool` return `Result(False, "a wall is in the way")` instead of calling `stand_up`, and ask Pupper to stand up. Then make it return `Result(True, "standing up")`, still without calling `stand_up`: nothing reaches the fake robot (no `[fake robot]` lines). What does Gemini tell you each time?

Write down what you saw: it is **Q1** in the lab document.

## 5. Put it back

This undoes your edits to `lab/basics.py`.

In [ ]:
!git checkout -- lab/basics.py && echo "lab/basics.py is back to the original"

From here on, the lab continues on your Pupper. Keep this notebook: it is the quickest place to try a change to what Gemini reads, before you try it on the robot.